# Column Domain Classifier — RandomForest vs GradientBoosting vs XGBoost

**DataGuard AI — Phase D.1**

We compare three classifiers on the same merged training data (`training_data_merged.csv`) to see which gives the best real-world accuracy:

- **RandomForest** (current baseline, column-risk-v3)
- **GradientBoosting** (scikit-learn)
- **XGBoost** (gradient boosting on sterols)

We use 5-fold stratified cross-validation on the same train/test split for all three.

In [25]:
import sys
from pathlib import Path

# Ensure backend root is on path so `app.ml...` imports work
backend_root = Path.cwd().resolve()
while backend_root.name != "backend" and backend_root.parent != backend_root:
    backend_root = backend_root.parent
if str(backend_root) not in sys.path:
    sys.path.insert(0, str(backend_root))
print("Backend root:", backend_root)

Backend root: C:\Users\DIPJYOTI\Downloads\DataGuard-AI-with-auth\merged\backend


In [26]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import time

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("XGBoost not installed")

from app.ml.feature_extractor import prepare_training_dataframe, build_preprocessor
from app.ml.model import create_column_classifier_pipeline

## Load & Prepare Data

In [27]:
ARTIFACTS = backend_root / "app" / "ml" / "artifacts"
DATA_FILE = ARTIFACTS / "training_data_merged.csv"

df_raw = pd.read_csv(DATA_FILE)
df = prepare_training_dataframe(df_raw)

from sklearn.preprocessing import LabelEncoder

X = df.drop(columns=["target_domain"])

# Original target labels
y = df["target_domain"]

# Encode target labels for XGBoost
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Total rows: {len(df)}")
print(f"Features: {len(X.columns)}")
print(f"Classes: {len(label_encoder.classes_)}")
print()
print("Class mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{i} -> {label}")

print()
print("Class distribution:")
print(y.value_counts().to_string())

Total rows: 6548
Features: 31
Classes: 12

Class mapping:
0 -> Categorical
1 -> Contact Information
2 -> Demographic
3 -> Financial
4 -> Identifier
5 -> Location
6 -> Numerical / Measurement
7 -> Other
8 -> Personal Information
9 -> Technical / System
10 -> Temporal
11 -> Text

Class distribution:
target_domain
Text                       910
Numerical / Measurement    872
Categorical                826
Identifier                 725
Personal Information       657
Temporal                   585
Contact Information        372
Location                   352
Demographic                322
Financial                  316
Technical / System         311
Other                      300


In [28]:
RANDOM_STATE = 42
TEST_SIZE = 0.20

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

Train: 5238 | Test: 1310


## Build the Three Pipelines

All three reuse the same `build_preprocessor()` (TF-IDF over column name + StandardScaler over numeric stats). Only the classifier changes.

In [29]:
from sklearn.pipeline import Pipeline

preprocessor_rf = build_preprocessor()
preprocessor_gb = build_preprocessor()

pipe_rf = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("classifier", RandomForestClassifier(
        n_estimators=120, max_depth=16, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1,
    )),
])

pipe_gb = Pipeline([
    ("preprocessor", preprocessor_gb),
    ("classifier", GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        random_state=RANDOM_STATE,
    )),
])

if HAS_XGBOOST:
    preprocessor_xgb = build_preprocessor()
    pipe_xgb = Pipeline([
        ("preprocessor", preprocessor_xgb),
        ("classifier", XGBClassifier(
            n_estimators=300, 
            learning_rate=0.1, 
            max_depth=6,
            subsample=0.9, 
            colsample_bytree=0.9,
            objective="multi:softprob", 
            num_class=len(label_encoder.classes_),
            eval_metric="mlogloss",
            random_state=RANDOM_STATE, 
            n_jobs=-1,
            tree_method="hist", 
            verbosity=0,
        )),
    ])
else:
    pipe_xgb = None

print("Pipelines built:")
print(" - RandomForest")
print(" - GradientBoosting")
print(" - XGBoost" if HAS_XGBOOST else " - XGBoost: NOT INSTALLED")

Pipelines built:
 - RandomForest
 - GradientBoosting
 - XGBoost


## 5-Fold Cross-Validation on Training Data

We run cross-validation on the training split only to compare generalization performance fairly.

In [30]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def evaluate(name, pipe):
    print(f"Evaluating {name}...")
    t0 = time.time()
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
    t_cv = time.time() - t0
    print(f"  CV accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})  [{t_cv:.1f}s]")
    return scores.mean(), scores.std()

results = {}
results["RandomForest"] = evaluate("RandomForest", pipe_rf)
results["GradientBoosting"] = evaluate("GradientBoosting", pipe_gb)
if pipe_xgb is not None:
    results["XGBoost"] = evaluate("XGBoost", pipe_xgb)

Evaluating RandomForest...
  CV accuracy: 0.8331 (+/- 0.0128)  [5.9s]
Evaluating GradientBoosting...
  CV accuracy: 0.9286 (+/- 0.0084)  [87.4s]
Evaluating XGBoost...
  CV accuracy: 0.9137 (+/- 0.0064)  [19.6s]


## Held-Out Test Set Evaluation

Now fit each pipeline on the full training set and evaluate on the untouched test set.

In [31]:
def fit_and_test(name, pipe):
    print(f"Fitting {name} on full training set...")
    t0 = time.time()
    pipe.fit(X_train, y_train)
    t_fit = time.time() - t0
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1_w = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    print(f"  Fit time: {t_fit:.1f}s")
    print(f"  Test accuracy:   {acc:.4f}")
    print(f"  Test F1 (weighted): {f1_w:.4f}")
    print()
    return pipe, acc, f1_w, classification_report(y_test, y_pred, output_dict=True, zero_division=0)

trained = {}
trained["RandomForest"] = fit_and_test("RandomForest", pipe_rf)
trained["GradientBoosting"] = fit_and_test("GradientBoosting", pipe_gb)
if pipe_xgb is not None:
    trained["XGBoost"] = fit_and_test("XGBoost", pipe_xgb)

Fitting RandomForest on full training set...
  Fit time: 0.3s
  Test accuracy:   0.8496
  Test F1 (weighted): 0.8488

Fitting GradientBoosting on full training set...
  Fit time: 55.2s
  Test accuracy:   0.9305
  Test F1 (weighted): 0.9305

Fitting XGBoost on full training set...
  Fit time: 7.5s
  Test accuracy:   0.9198
  Test F1 (weighted): 0.9197



## Comparison Table

In [32]:
rows = []
for name, (pipe, acc, f1w, report) in trained.items():
    rows.append({
        "Model": name,
        "CV mean": round(results[name][0], 4),
        "CV std": round(results[name][1], 4),
        "Test accuracy": round(acc, 4),
        "Test F1 (weighted)": round(f1w, 4),
    })

results_df = pd.DataFrame(rows).sort_values("Test accuracy", ascending=False)
print(results_df.to_string(index=False))

           Model  CV mean  CV std  Test accuracy  Test F1 (weighted)
GradientBoosting   0.9286  0.0084         0.9305              0.9305
         XGBoost   0.9137  0.0064         0.9198              0.9197
    RandomForest   0.8331  0.0128         0.8496              0.8488


In [33]:
best_name = results_df.iloc[0]["Model"]
best_report = trained[best_name][3]

print(f"Best model: {best_name}")
print()

report_df = pd.DataFrame({
    k: v for k, v in best_report.items()
    if k not in ("accuracy", "macro avg", "weighted avg")
}).T
report_df = report_df[["precision", "recall", "f1-score", "support"]].sort_values("f1-score")
print(report_df.round(3).to_string())

Best model: GradientBoosting

    precision  recall  f1-score  support
3       0.947   0.857     0.900     63.0
0       0.908   0.897     0.902    165.0
10      0.913   0.897     0.905    117.0
6       0.905   0.926     0.915    175.0
5       0.928   0.914     0.921     70.0
9       0.949   0.903     0.926     62.0
11      0.886   0.984     0.932    182.0
8       0.960   0.917     0.938    132.0
7       0.966   0.933     0.949     60.0
1       0.986   0.933     0.959     75.0
4       0.953   0.979     0.966    145.0
2       1.000   0.969     0.984     64.0


In [34]:
import json
import joblib
from datetime import datetime, timezone
import sklearn

best_pipe = trained[best_name][0]
best_acc = trained[best_name][1]

MODEL_VERSION = "column-risk-v4"
model_path = ARTIFACTS / f"{MODEL_VERSION}.joblib"
meta_path = ARTIFACTS / f"metadata_{MODEL_VERSION}.json"

joblib.dump(best_pipe, model_path)

metadata = {
    "model_version": MODEL_VERSION,
    "algorithm": best_name,
    "sklearn_version": sklearn.__version__,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "training_source": DATA_FILE.name,
    "total_samples": int(len(df)),
    "num_classes": int(y.nunique()),
    "classes": sorted(list(best_pipe.classes_)),
    "metrics": {
        "test_accuracy": round(best_acc, 4),
        "cv_accuracy_mean": round(results[best_name][0], 4),
        "cv_accuracy_std": round(results[best_name][1], 4),
        "classification_report": best_report,
    },
}

with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"Model saved: {model_path}  ({model_path.stat().st_size / 1024:.0f} KB)")
print(f"Metadata: {meta_path}")
print(f"Best model: {best_name}  (test accuracy {best_acc:.4f})")

Model saved: C:\Users\DIPJYOTI\Downloads\DataGuard-AI-with-auth\merged\backend\app\ml\artifacts\column-risk-v4.joblib  (7480 KB)
Metadata: C:\Users\DIPJYOTI\Downloads\DataGuard-AI-with-auth\merged\backend\app\ml\artifacts\metadata_column-risk-v4.json
Best model: GradientBoosting  (test accuracy 0.9305)
